# Lecture 13 Activity – Which model should decide the disputed essays?

QSS 20 · Modern Statistical Computing · Fall 2026

## The question

The lecture fitted three classifiers on five representations of the
Federalist essays. On the ten style words, all three found Madison's
known essays and agreed on ten of the 12 disputed essays. On the wide
TF-IDF matrices, logistic regression scored 1.0 on the training essays
and little above the baseline on the test essays. A report has to name
one model, one representation, and the evidence for both.

> **Which classifier, on which representation, should decide the 12
> disputed essays, and what in the held-out scores supports that choice?**

### How to work

Work in pairs for 30 minutes. Submit the best $k$, the three models'
training and test scores, the three classification reports, the 12
verdicts, and a short claim.
If you run out of time, finish Task 6 after class.

- The row unit is **one essay**. The setup reads the 85 files, labels the
  authors as the lecture did, builds the TF-IDF matrix with the stop list
  kept, and takes the ten style words as columns of it.
- Split the known essays once, with the lecture's settings:
  `test_size=0.25`, `random_state=42`, `stratify=`. Fit on the training
  rows; score on the test data; refit on all 65 only for the verdicts.
- In the starter cells, replace each `...` with code.

In [1]:
import os
os.environ['OMP_NUM_THREADS'] = '1'

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import (
    GridSearchCV, cross_val_score, train_test_split)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree

RAW = Path('../data/raw')

rows = []
for n in range(1, 86):
    path = RAW / 'federalist' / f'fp{n:02d}.txt'
    rows.append({'essay': n,
                 'text': path.read_text(encoding='utf-8')})
essays = pd.DataFrame(rows)

authors = {
    'Hamilton': ([1, 6, 7, 8, 9, 11, 12, 13, 15, 16, 17]
                 + list(range(21, 37)) + [59, 60, 61]
                 + list(range(65, 86))),
    'Madison': [10, 14] + list(range(37, 49)),
    'Jay': [2, 3, 4, 5, 64],
    'Joint': [18, 19, 20],
    'Disputed': list(range(49, 59)) + [62, 63],
}
label = {n: author for author, ns in authors.items() for n in ns}
essays['author'] = essays['essay'].map(label)
author = essays.set_index('essay')['author']

# The one matrix of the lecture: TF-IDF, stop list kept, as a DataFrame.
tfidf_vec = TfidfVectorizer(min_df=2)
tfidf = pd.DataFrame(tfidf_vec.fit_transform(essays['text']).toarray(),
                     columns=tfidf_vec.get_feature_names_out(),
                     index=essays['essay'])
style_words = ['upon', 'whilst', 'while', 'there', 'on', 'although',
               'always', 'commonly', 'consequently', 'enough']
style = tfidf[style_words]

known_ids = author.index[author.isin(['Hamilton', 'Madison'])]
disputed_ids = author.index[author == 'Disputed']
X = style.loc[known_ids]
y = author.loc[known_ids]
X_disputed = style.loc[disputed_ids]
classes = ['Hamilton', 'Madison']

# Check: 65 known rows, 12 disputed rows, ten columns.
X.shape, X_disputed.shape, y.value_counts().to_dict()

((65, 10), (12, 10), {'Hamilton': 51, 'Madison': 14})

## Task 1 · The baseline and the split

Compute the baseline accuracy of a model that answers Hamilton for every
known essay. Then split `X` and `y` into training and test data with the
lecture's settings and print the author counts in each part.

How many Madison essays does the test set hold, and what would a model
that never answers Madison score on it?

In [2]:
baseline = y.value_counts(normalize=True).max()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)
print(round(baseline, 3))
print(y_train.value_counts().to_dict(), y_test.value_counts().to_dict())

0.785
{'Hamilton': 38, 'Madison': 10} {'Hamilton': 13, 'Madison': 4}


The baseline is 0.785. The training data hold 38 Hamilton and 10 Madison
essays; the test data hold 13 Hamilton and 4 Madison. A model that never
answers Madison scores 13 of 17 on the test data, 0.765, which is what
logistic regression on the 4,168-column matrix scored in the lecture.

## Task 2 · The best $k$

Build a pipeline of `StandardScaler` and `KNeighborsClassifier`, and use
`GridSearchCV` with five folds on the training data to choose $k$ among
the odd numbers from 1 to 19. Report the best $k$, its cross-validated
accuracy, and the accuracy on the test data. Then print the
cross-validated accuracy for every $k$.

Where does the accuracy reach the baseline, and why does it fall there?

In [3]:
knn_search = GridSearchCV(
    make_pipeline(StandardScaler(), KNeighborsClassifier()),
    param_grid={'kneighborsclassifier__n_neighbors': range(1, 20, 2)},
    cv=5)
knn_search.fit(X_train, y_train)
print(knn_search.best_params_, round(knn_search.best_score_, 3))
print('test', round(knn_search.score(X_test, y_test), 3))

{'kneighborsclassifier__n_neighbors': 1} 0.958
test 0.941


In [4]:
pd.Series(knn_search.cv_results_['mean_test_score'],
          index=range(1, 20, 2)).round(3)

1     0.958
3     0.958
5     0.938
7     0.938
9     0.938
11    0.898
13    0.898
15    0.833
17    0.791
19    0.791
dtype: float64

The best $k$ is 1, with a cross-validated accuracy of 0.958 and 0.941 on
the 17 test data; $k = 3$ scores the same across the folds, and the
scores stay above 0.93 to $k = 9$. They reach 0.791, the training-row
baseline, at $k = 17$: the training data hold only 10 Madison essays, so
once $k$ is large next to that number, Hamilton essays outvote Madison's
among any essay's neighbors and the model answers Hamilton for everyone.

## Task 3 · Training accuracy against test accuracy

Build the lecture's logistic pipeline (`StandardScaler`, then
`LogisticRegression(max_iter=2000)`) and a
`DecisionTreeClassifier(max_depth=2, random_state=0)`, and fit both on
the training data. For kNN (the search of Task 2), the tree, and logistic
regression, print three numbers with `.score` and `cross_val_score`: the
accuracy on the training data, the five-fold mean on the training data,
and the accuracy on the test data.

Which of the three numbers would you report, and what does the gap
between the first and the other two say?

In [5]:
logit = make_pipeline(StandardScaler(),
                      LogisticRegression(max_iter=2000))
logit.fit(X_train, y_train)
tree = DecisionTreeClassifier(max_depth=2, random_state=0)
tree.fit(X_train, y_train)
models = {'knn': knn_search.best_estimator_, 'tree': tree,
          'logistic': logit}
for name, model in models.items():
    folds = cross_val_score(model, X_train, y_train, cv=5)
    print(f'{name:9s} train {model.score(X_train, y_train):.3f}'
          f'   five folds {folds.mean():.3f}'
          f'   test {model.score(X_test, y_test):.3f}')

knn       train 1.000   five folds 0.958   test 0.941


tree      train 1.000   five folds 1.000   test 0.941
logistic  train 1.000   five folds 0.980   test 0.941


All three models score 1.0 on the training data: kNN with $k = 1$
because every training essay is its own nearest neighbor, the tree
because one cut on `upon` sorts the 48 essays, logistic regression
because ten scaled columns can separate 48 rows. The five folds (0.958,
1.0, 0.980) and the test data (0.941 for all three) are the numbers to
report. The gap says that a training score measures how well the model
repeats what it saw; here it is small, so the models learned the writers
and not the essays, but the training score alone could not have told us
that.

## Task 4 · Three classification reports

For kNN, the tree, and logistic regression, print the confusion matrix
on the test data with Hamilton first, then `classification_report`, and
list the essay numbers each model gets wrong.

Do the three models make the same mistakes, and which net is each?

In [6]:
for name, model in models.items():
    out = model.predict(X_test)
    print(name, '  wrong:', list(X_test.index[out != y_test]))
    print(pd.DataFrame(confusion_matrix(y_test, out, labels=classes),
                       index=classes, columns=classes))
    print(classification_report(y_test, out))

knn   wrong: [32]
          Hamilton  Madison
Hamilton        12        1
Madison          0        4
              precision    recall  f1-score   support

    Hamilton       1.00      0.92      0.96        13
     Madison       0.80      1.00      0.89         4

    accuracy                           0.94        17
   macro avg       0.90      0.96      0.92        17
weighted avg       0.95      0.94      0.94        17

tree   wrong: [38]
          Hamilton  Madison
Hamilton        13        0
Madison          1        3
              precision    recall  f1-score   support

    Hamilton       0.93      1.00      0.96        13
     Madison       1.00      0.75      0.86         4

    accuracy                           0.94        17
   macro avg       0.96      0.88      0.91        17
weighted avg       0.95      0.94      0.94        17

logistic   wrong: [32]
          Hamilton  Madison
Hamilton        12        1
Madison          0        4
              precision    recall 

No. kNN and logistic regression miss the same single essay, 32, a
Hamilton essay they call Madison's: Madison recall 1.00 and precision
0.80, the wide net. The tree misses only essay 38, a Madison essay it
calls Hamilton's: recall 0.75 and precision 1.00, the small net. All
three score 0.94, above the 0.765 that a model which never answers
Madison would score, but the accuracy hides that they fail in opposite
directions. Seventeen test essays cannot separate the two: each mistake
is one essay.

## Task 5 · The twelve verdicts

Refit each model on all 65 known rows: kNN with the best $k$ from Task 2,
the tree with `max_depth=2`, and the logistic pipeline. Label the 12
disputed essays with each, add the logistic probability of Madison, and
count the essays on which the models disagree.

Which essays divide the models, and what in their style columns explains
the split?

In [7]:
best_k = knn_search.best_params_['kneighborsclassifier__n_neighbors']
knn_all = make_pipeline(StandardScaler(),
                        KNeighborsClassifier(n_neighbors=best_k))
tree_all = DecisionTreeClassifier(max_depth=2, random_state=0)
logit_all = make_pipeline(StandardScaler(),
                          LogisticRegression(max_iter=2000))
for model in [knn_all, tree_all, logit_all]:
    model.fit(X, y)
verdicts = pd.DataFrame({
    'knn': knn_all.predict(X_disputed),
    'tree': tree_all.predict(X_disputed),
    'logistic': logit_all.predict(X_disputed),
    'p_madison': logit_all.predict_proba(X_disputed)[:, 1].round(2)},
    index=X_disputed.index)
split = verdicts[['knn', 'tree', 'logistic']].nunique(axis=1) > 1
print('essays with a disagreement:', list(verdicts.index[split]))
verdicts

essays with a disagreement: [54, 55]


,knn,tree,logistic,p_madison
essay,,,,
49,Madison,Madison,Madison,0.99
50,Madison,Madison,Madison,0.92
51,Madison,Madison,Madison,1.00
52,Madison,Madison,Madison,0.93
53,Madison,Madison,Madison,0.84
54,Madison,Hamilton,Madison,0.99
55,Madison,Madison,Hamilton,0.32
56,Madison,Madison,Madison,0.93
57,Madison,Madison,Madison,0.99


In [8]:
X_disputed.loc[split, ['upon', 'there', 'on']].round(4)

,upon,there,on
essay,,,
54,0.0091,0.0038,0.0642
55,0.0000,0.0201,0.0326


Two essays divide the models. Essay 54 is Hamilton for the tree alone:
refitted on all 65 essays, the tree cuts `upon` at a weight of 0.0084,
and essay 54 sits at 0.0091, one use of the word above the line, while
its `on` weight is Madison's. Essay 55 is Hamilton for logistic
regression (0.32) and Madison for the other two: it never uses `upon`,
which the tree decides on, but it uses `there` at Hamilton's rate and
`on` at half of Madison's, and the logistic weights add those up. The
other ten essays are Madison for all three models.

## Task 6 · Make a claim

Write four or five sentences. Name the model and the representation you
would report for the 12 disputed essays and the held-out evidence that
supports the choice. Say what the two disagreements mean for the report.
Then say why logistic regression on the 4,168-column matrix of the lecture
cannot settle the question even though it scored 1.0 on the training
essays.

Report logistic regression on the ten style columns: it finds all four
Madison test essays, scores 0.941 on the test data against a 0.785
baseline, and its five folds average 0.98. The tree scores the same on
the test data with one word, which makes it the easiest model to explain,
but a model built on one cut has no margin for an essay that sits at the
cut, as essay 54 shows. The two disagreements are the report's honest
edge: essays 54 and 55 should be listed as Madison with a stated doubt,
and the probability 0.32 for essay 55 belongs in the table, because the
models that call it Madison do so on `upon` alone. The 4,168-column
model cannot settle anything because its 1.0 is a training score: on the
test essays it scored 0.76, below the 0.785 baseline, and its
classification report shows that it never answers Madison. A score earned
on the rows the fit saw is no evidence about rows it did not.

## Checks for a claim built on a classifier

- Compare every accuracy with the baseline of the same rows, and read
  the test score, not the training score: the wide matrix scored 1.0 on
  the training data and 0.76 on the test data.
- Read recall and precision before the accuracy. Two models can score
  0.941 and fail in opposite directions.
- With 17 test essays, one essay is 0.059 of accuracy. Report the
  cross-validated scores beside the single split.
- Where models disagree, report the disagreement and the probability, not
  the majority vote.